# Significance, Categorization, and Likelihood Analysis — companion notebook

Physics 129 discussion section (11/04). One section per page of the slides (`likelihood_notes.pdf`).
Each section has sliders: move them and watch the plot change.

**How to use:** run all cells (*Run → Run All Cells*, or *Runtime → Run all* in Colab), then scroll to the page you want.
Needs `numpy`, `scipy`, `matplotlib`, `ipywidgets` (all preinstalled on Colab and most JupyterHubs; otherwise `pip install ipywidgets`).

| Page | Topic | Page | Topic |
|---|---|---|---|
| 2 | Significance | 10 | Categories are the input to the measurement |
| 3 | Significance: purely statistical uncertainty | 11 | Likelihood analysis |
| 4 | Hypothesis and observation | 12 | Negative log-likelihood |
| 5 | Which one has a genuine signal? | 13 | Likelihood analysis: signal strength |
| 6 | None of them is a signal | 14 | Negative log-likelihood in $\mu$ |
| 7 | $p$-value and significance | 15 | Likelihood with multiple categories |
| 8 | Confidence limit | 16 | Signal strength from the NLL |
| 9 | Categorization | | |

In [ ]:
# Setup: run this first.
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy import stats, optimize
from scipy.special import gammaln
import ipywidgets as w
from ipywidgets import interact

BB, DR, GRAY = '#003262', '#912328', '#b8b8b8'      # slide colors
CATC = ['#1f5fa8', '#d9822b', '#3a9b48']             # category colors
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

_style, _wide = {'description_width': 'initial'}, w.Layout(width='440px')
def fslider(v, lo, hi, step, desc, cont=True):
    return w.FloatSlider(value=v, min=lo, max=hi, step=step, description=desc,
                         continuous_update=cont, style=_style, layout=_wide)
def islider(v, lo, hi, desc, step=1, cont=True):
    return w.IntSlider(value=v, min=lo, max=hi, step=step, description=desc,
                       continuous_update=cont, style=_style, layout=_wide)
def logslider(v, lo, hi, desc, cont=True):
    return w.FloatLogSlider(value=v, base=10, min=np.log10(lo), max=np.log10(hi), step=0.01,
                            description=desc, continuous_update=cont, style=_style, layout=_wide)
def choice(options, v, desc):
    return w.SelectionSlider(options=options, value=v, description=desc, style=_style, layout=_wide)

def nll_poisson(lam, k):
    # -ln L(lambda) = lambda - k ln(lambda) + ln k!   (gammaln(k+1) = ln k!)
    return lam - k * np.log(lam) + gammaln(k + 1)

# ---- m_gammagamma model used on several pages (same numbers as the slides) ----
LO, HI = 105.0, 160.0          # mass window [GeV]
MH, SIGMA_M = 125.0, 1.6       # signal peak position and width [GeV]

def bkg_expected(N, binw=2.5, tau=30.0):
    # Expected background per bin when N events are drawn from an exponential in [LO, HI].
    edges = np.arange(LO, HI + 1e-9, binw)
    cdf = lambda m: 1 - np.exp(-(m - LO) / tau)
    return edges, N * (cdf(edges[1:]) - cdf(edges[:-1])) / cdf(HI)

def bkg_curve(x, N, binw=2.5, tau=30.0):
    # Smooth background expectation in events / bin (the dashed line on the slides).
    return N * binw * np.exp(-(x - LO) / tau) / (tau * (1 - np.exp(-(HI - LO) / tau)))

# Three categories (page 10/15): peak heights per GeV, from the slide's illustration.
SIG_H = 12 * np.array([5, 5, 4]) / 14          # signal Gaussian heights
BKG_H = 80 * np.array([1.5, 9, 70]) / 80.5     # background exponential heights at 105 GeV
TAU_C = 28.0
EDGES1 = np.arange(LO, HI + 1e-9, 1.0)         # 1 GeV bins
CENT1 = 0.5 * (EDGES1[1:] + EDGES1[:-1])

def category_templates(separation=1.0, lumi=1.0):
    # Expected signal s[c, i] and background b[c, i] in 1 GeV bins for the 3 categories.
    # separation = 0: every category has the same s/b (no discrimination); 1: slide values.
    fs = SIG_H / SIG_H.sum()
    g = (1 - separation) * fs + separation * BKG_H / BKG_H.sum()
    sig_amp, bkg_amp = lumi * SIG_H.sum() * fs, lumi * BKG_H.sum() * g
    sig_shape = (stats.norm.cdf(EDGES1[1:], MH, SIGMA_M) - stats.norm.cdf(EDGES1[:-1], MH, SIGMA_M)) \
                * SIGMA_M * np.sqrt(2 * np.pi)
    bkg_shape = TAU_C * (np.exp(-(EDGES1[:-1] - LO) / TAU_C) - np.exp(-(EDGES1[1:] - LO) / TAU_C))
    return np.outer(sig_amp, sig_shape), np.outer(bkg_amp, bkg_shape)

print('Setup done.')

## Page 2 — Significance

Significance is the size of the signal in units of the background uncertainty:
$$Z = \frac{S}{\delta B}$$

**Try:**
- Keep $S$ fixed and make $\delta B$ larger. The same signal becomes less significant.
- Find two different $(S, \delta B)$ pairs that give the same $Z$.
- Tick *show tail* to see how much of the background-only curve lies beyond $B+S$ (this area returns on page 7).

In [ ]:
def significance(B=100.0, dB=10.0, S=30.0, show_tail=False):
    Z = S / dB
    x = np.linspace(B - 5 * dB, B + max(S + 3 * dB, 5 * dB), 500)
    y = np.exp(-(x - B) ** 2 / (2 * dB ** 2))
    fig, ax = plt.subplots(figsize=(8.5, 3.4))
    ax.fill_between(x, y, color=BB, alpha=.12)
    ax.plot(x, y, color=BB, lw=2.5)
    ax.plot([B, B], [0, 1], '--', color=BB)
    if show_tail:
        t = x >= B + S
        ax.fill_between(x[t], y[t], color=DR, alpha=.6)
        ax.text(B + S + dB * .3, .3, f'tail area = {stats.norm.sf(Z):.2g}', color=DR)
    ax.annotate('', xy=(B + dB, .45), xytext=(B, .45), arrowprops=dict(arrowstyle='<->', lw=1.5))
    ax.text(B + dB / 2, .49, r'$\delta B$', ha='center')
    ax.plot([B + S, B + S], [0, .75], color=DR, lw=2)
    ax.text(B + S, .78, '$B+S$', color=DR, ha='center')
    ax.annotate('', xy=(B + S, .12), xytext=(B, .12), arrowprops=dict(arrowstyle='<->', color=DR, lw=1.5))
    ax.text(B + .75 * S, .15, '$S$', color=DR, ha='center')
    ax.set_ylim(0, 1.15); ax.set_yticks([]); ax.set_xlabel('event yield')
    ax.set_title(f'$Z = S/\\delta B = {S:.0f}/{dB:.1f} = {Z:.2f}$')
    plt.show()

interact(significance, B=fslider(100, 20, 500, 1, 'B (background)'),
         dB=fslider(10, 1, 60, .5, 'δB (background uncertainty)'),
         S=fslider(30, 0, 150, 1, 'S (signal)'), show_tail=w.Checkbox(False, description='show tail'));

## Page 3 — Significance: purely statistical uncertainty

If the uncertainty is purely statistical, $B$ follows a Poisson distribution (close to normal for large $B$) with standard deviation $\sqrt{B}$:
$$\delta B = \sqrt{B}, \qquad Z = \frac{S}{\sqrt{B}} \quad (S \ll B), \qquad Z = \sqrt{2\left[(S+B)\ln(1+S/B) - S\right]} \quad \text{(general)}$$

**Try:**
- Make $B$ small (1–10). The Poisson bars are no longer a Gaussian.
- At $B = 100$, increase $S$ from 10 to 100. Where do the two formulas start to disagree?
- Multiply both $S$ and $B$ by 4 (e.g. more data). What happens to $Z$?

In [ ]:
def z_asimov(S, B):
    return np.sqrt(2 * ((S + B) * np.log1p(S / B) - S))

def stat_only(B=100.0, S=30.0):
    sB = np.sqrt(B)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw=dict(width_ratios=[1.3, 1]))
    n = np.arange(max(0, int(B - 5 * sB) - 2), int(max(B + S, B + 5 * sB)) + 4)
    a1.bar(n, stats.poisson.pmf(n, B), width=1.0 if len(n) < 80 else 1.05, color=GRAY, label=f'Poisson, mean {B:.3g}')
    x = np.linspace(n[0], n[-1], 400)
    a1.plot(x, stats.norm.pdf(x, B, sB), color=BB, lw=2, label=r'normal, $\sigma=\sqrt{B}$')
    top = stats.norm.pdf(B, B, sB)
    a1.axvline(B + S, color=DR, lw=2); a1.text(B + S, top * 1.05, ' $B+S$', color=DR)
    a1.set_xlabel('number of background events'); a1.set_yticks([]); a1.legend(fontsize=9, loc='upper left')
    a1.set_ylim(0, top * 1.25)
    Ss = np.linspace(0, max(3 * S, 5 * sB, 1), 300)
    a2.plot(Ss, Ss / sB, color=BB, lw=2, label=r'$S/\sqrt{B}$')
    a2.plot(Ss, z_asimov(Ss, B), color=DR, lw=2, label='general formula')
    a2.plot([S], [S / sB], 'o', color=BB); a2.plot([S], [z_asimov(S, B)], 'o', color=DR)
    a2.set_xlabel('S'); a2.set_ylabel('Z'); a2.legend(fontsize=9)
    a2.set_title(f'S/B = {S / B:.2f}', fontsize=10)
    fig.suptitle(f'$S/\\sqrt{{B}}$ = {S / sB:.2f}    general = {z_asimov(S, B):.2f}', y=1.0)
    plt.tight_layout(); plt.show()

interact(stat_only, B=logslider(100, 1, 1e4, 'B (background)'), S=fslider(30, 0, 300, 1, 'S (signal)'));

## Page 4 — Hypothesis and observation

**Hypothesis:** the expectation. **Observation:** a random sample from that hypothesis.

The gray bars are the Poisson probabilities for the hypothesis. Each red line is one observation.

**Try:**
- Change the seed with 4 observations. Each time you get a different set of numbers from the *same* hypothesis.
- Increase the number of observations to 1000 or 10000. The histogram of observations (red) approaches the hypothesis (gray).
- Compare the spread of the observations with $\sqrt{\text{mean}}$.

In [ ]:
def hypothesis(mean=100.0, n_obs=4, seed=1):
    rng = np.random.default_rng(seed)
    draws = rng.poisson(mean, n_obs)
    s = np.sqrt(mean)
    n = np.arange(max(0, int(mean - 4.5 * s)), int(mean + 4.5 * s) + 2)
    pmf = stats.poisson.pmf(n, mean)
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.vlines(n, 0, pmf, color=GRAY, lw=3 if len(n) < 120 else 1.5, label=f'hypothesis: Poisson, mean {mean:g}')
    if n_obs <= 12:
        for d in draws:
            ax.plot([d, d], [0, stats.poisson.pmf(d, mean)], color=DR, lw=2.5)
            ax.text(d, stats.poisson.pmf(d, mean) * 1.04, str(d), color=DR, ha='center', fontsize=10)
    else:
        vals, cnt = np.unique(draws, return_counts=True)
        ax.plot(vals, cnt / n_obs, 'o', color=DR, ms=4, label=f'{n_obs} observations (fraction per value)')
    ax.set_xlim(n[0] - 1, n[-1] + 1); ax.set_ylim(0, pmf.max() * 1.35)
    ax.set_xlabel('number of events'); ax.set_yticks([]); ax.legend(loc='upper right', fontsize=9)
    sd = f', std {draws.std(ddof=1):.1f}' if n_obs > 1 else ''
    ax.set_title(f'observations: mean {draws.mean():.1f}{sd}   (hypothesis: mean {mean:g}, $\\sqrt{{\\rm mean}}$ = {s:.1f})', fontsize=10)
    plt.show()

interact(hypothesis, mean=fslider(100, 2, 400, 1, 'hypothesis mean'),
         n_obs=choice([1, 2, 4, 8, 12, 100, 1000, 10000], 4, 'number of observations'),
         seed=islider(1, 0, 200, 'random seed'));

## Page 5 — Which one has a genuine signal?

Four pseudo-experiments: $m_{\gamma\gamma}$ of 1000 events each, drawn from an exponential (dashed line = background expectation).
The arrow marks the largest local excess in a ~5 GeV window, with its local significance.

**Quiz mode:** set *signal events* above zero. The signal is hidden in **one** random panel (at *signal mass*).
Guess which one, then tick *reveal*.

**Try:**
- With 0 signal events, click through seeds. How often do you see a "local Z" above 2? Above 3?
- How many signal events do you need before you can reliably pick the right panel?
- Increase the number of events (more data) with the signal fixed as a fraction. Does the true signal become easier to find?

In [ ]:
def which_one(seed=0, n_signal=0, m_signal=125.0, N_events=1000, bin_width=2.5, reveal=False):
    rng = np.random.default_rng(seed)
    edges, exp = bkg_expected(N_events, bin_width)
    c = 0.5 * (edges[1:] + edges[:-1])
    hidden = rng.integers(4)
    allc = []
    for i in range(4):
        counts = rng.multinomial(N_events, exp / exp.sum())
        if i == hidden and n_signal > 0:
            counts = counts + np.histogram(rng.normal(m_signal, SIGMA_M, n_signal), edges)[0]
        allc.append(counts)
    ymax = max(cc.max() + np.sqrt(cc.max()) for cc in allc) * 1.3
    nw = max(1, int(round(5.0 / bin_width)))
    fig, axs = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
    xx = np.linspace(LO, HI, 200)
    for i, (ax, counts) in enumerate(zip(axs.flat, allc)):
        ax.plot(xx, bkg_curve(xx, N_events, bin_width), '--', color=BB, lw=1.8)
        ax.errorbar(c, counts, yerr=np.sqrt(counts), fmt='o', ms=3, color='k', lw=.8)
        n_w = np.convolve(counts, np.ones(nw), 'valid'); b_w = np.convolve(exp, np.ones(nw), 'valid')
        z = (n_w - b_w) / np.sqrt(b_w); j = int(np.argmax(z))
        mc = edges[j] + nw * bin_width / 2
        tip = counts[j:j + nw].max() + np.sqrt(counts[j:j + nw].max()) + .02 * ymax
        ax.annotate('', xy=(mc, tip), xytext=(mc, tip + .16 * ymax),
                    arrowprops=dict(arrowstyle='->', color=DR, lw=1.8))
        ax.text(mc, tip + .17 * ymax, f'local Z = {z[j]:.1f}', color=DR, ha='center', fontsize=9)
        ax.text(.97, .93, 'ABCD'[i], transform=ax.transAxes, ha='right', va='top', fontsize=14, weight='bold')
        if reveal:
            has = (i == hidden and n_signal > 0)
            ax.set_title('SIGNAL here' if has else 'background only', color=DR if has else 'gray', fontsize=10)
            if has:
                ax.plot(xx, bkg_curve(xx, N_events, bin_width) + n_signal * bin_width * stats.norm.pdf(xx, m_signal, SIGMA_M),
                        color=DR, lw=1.2, alpha=.7)
        ax.set_ylim(0, ymax); ax.set_xlim(LO, HI)
    for ax in axs[1]: ax.set_xlabel(r'$m_{\gamma\gamma}$ [GeV]')
    for ax in axs[:, 0]: ax.set_ylabel(f'events / {bin_width:g} GeV')
    fig.suptitle('Which of A, B, C, D has a genuine signal?' if not reveal else
                 ('None of them is a signal.' if n_signal == 0 else f'Signal ({n_signal} events) was in panel {"ABCD"[hidden]}.'))
    plt.tight_layout(); plt.show()

interact(which_one, seed=islider(0, 0, 500, 'random seed'),
         n_signal=islider(0, 0, 150, 'signal events (hidden in one panel)', step=5),
         m_signal=fslider(125, 110, 155, .5, 'signal mass [GeV]'),
         N_events=choice([250, 500, 1000, 2000, 5000, 10000], 1000, 'background events'),
         bin_width=choice([1.0, 2.5, 5.0], 2.5, 'bin width [GeV]'),
         reveal=w.Checkbox(False, description='reveal'));

## Page 6 — None of them is a signal

The four plots on the slide were picked out of **20000** background-only pseudo-experiments, for their pronounced local excesses.
If you look in many places (many masses) and many plots, large fluctuations appear somewhere. This is the *look-elsewhere effect*.

Left: distribution of the local $Z$ in **one fixed window** (at 125 GeV) and of the **largest** local $Z$ anywhere in the spectrum.
Right: the most extreme background-only toy among the ones generated.

**Try:**
- Increase the number of toys. How large does the "best" fake peak get?
- Compare the fraction of toys with $Z > 3$ in the fixed window and anywhere.
- Change the window width. Narrower windows = more places to look.

In [ ]:
def look_elsewhere(n_toys=1000, N_events=1000, window_GeV=5.0, seed=1):
    rng = np.random.default_rng(seed)
    binw = 2.5
    edges, exp = bkg_expected(N_events, binw)
    nw = int(round(window_GeV / binw))
    counts = rng.multinomial(N_events, exp / exp.sum(), size=n_toys)
    cs = np.cumsum(np.pad(counts, ((0, 0), (1, 0))), axis=1)
    n_w = cs[:, nw:] - cs[:, :-nw]
    be = np.cumsum(np.pad(exp, (1, 0))); b_w = be[nw:] - be[:-nw]
    z = (n_w - b_w) / np.sqrt(b_w)
    zmax = z.max(axis=1)
    wc = edges[:len(b_w)] + nw * binw / 2
    z_fix = z[:, np.argmin(abs(wc - MH))]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.9))
    bins = np.arange(-4, 6.01, .25)
    a1.hist(z_fix, bins, density=True, color=BB, alpha=.45, label='fixed window at 125 GeV')
    a1.hist(zmax, bins, density=True, histtype='step', color=DR, lw=2, label='largest anywhere')
    xg = np.linspace(-4, 6, 300); a1.plot(xg, stats.norm.pdf(xg), 'k:', lw=1, label='standard normal')
    a1.set_xlabel('local Z'); a1.set_yticks([]); a1.legend(fontsize=8.5, loc='upper left')
    a1.set_title(f'Z>3: {np.mean(z_fix > 3):.2%} (fixed)  vs  {np.mean(zmax > 3):.2%} (anywhere)', fontsize=10)
    k = int(np.argmax(zmax)); cnt = counts[k]; c = 0.5 * (edges[1:] + edges[:-1])
    xx = np.linspace(LO, HI, 200)
    a2.plot(xx, bkg_curve(xx, N_events, binw), '--', color=BB, lw=1.8)
    a2.errorbar(c, cnt, yerr=np.sqrt(cnt), fmt='o', ms=3, color='k', lw=.8)
    j = int(np.argmax(z[k])); mc = wc[j]; top = cnt[j:j + nw].max()
    a2.annotate('', xy=(mc, top + np.sqrt(top) + 2), xytext=(mc, top * 1.35 + 5), arrowprops=dict(arrowstyle='->', color=DR, lw=1.8))
    a2.set_ylim(0, cnt.max() * 1.45 + 5); a2.set_xlabel(r'$m_{\gamma\gamma}$ [GeV]'); a2.set_ylabel('events / 2.5 GeV')
    a2.set_title(f'most extreme of {n_toys} toys: local Z = {zmax[k]:.1f} at {mc:.1f} GeV (no signal!)', fontsize=10)
    plt.tight_layout(); plt.show()

interact(look_elsewhere, n_toys=choice([1, 4, 10, 100, 1000, 5000, 20000], 1000, 'number of toys'),
         N_events=choice([250, 1000, 5000], 1000, 'events per toy'),
         window_GeV=choice([2.5, 5.0, 7.5, 10.0], 5.0, 'window width [GeV]'),
         seed=islider(1, 0, 200, 'random seed', cont=False));

## Page 7 — $p$-value and significance

$p$-value: the probability, assuming the background-only hypothesis $H_0$, to find data of equal or lesser compatibility with $H_0$ than the observed data.
It is converted into a significance $Z$, an equivalent number of Gaussian standard deviations:
$$Z = \Phi^{-1}(1-p)$$
**The conversion using the normal distribution is just a convention.** The data themselves do not have to be Gaussian. Discovery: $Z = 5$.

**Try (first widget):** move $Z$ and read off $p$; check the table on the slide ($Z = 5 \to p = 2.87\times10^{-7}$). Switch to two-sided.
**Try (second widget):** set the $p$-value directly and read off $Z$. What $Z$ is $p = 0.05$? $p = 0.01$?

In [ ]:
def tail_plot(Z=3.0, sided='one-sided'):
    p = stats.norm.sf(Z) * (2 if sided == 'two-sided' else 1)
    x = np.linspace(-max(4.5, Z + 1), max(4.5, Z + 1), 800); y = stats.norm.pdf(x)
    fig, ax = plt.subplots(figsize=(8.5, 3.2))
    ax.plot(x, y, color=BB, lw=2.5)
    ax.fill_between(x[x >= Z], y[x >= Z], color=DR, alpha=.7)
    if sided == 'two-sided':
        ax.fill_between(x[x <= -Z], y[x <= -Z], color=DR, alpha=.7)
    ax.axvline(Z, color=DR, lw=1)
    rows = '\n'.join(f'Z={z}:  p={stats.norm.sf(z):.3g}' for z in range(1, 6))
    ax.text(.02, .95, 'one-sided table\n' + rows, transform=ax.transAxes, va='top', fontsize=8.5, family='monospace')
    ax.set_yticks([]); ax.set_ylim(0, .45); ax.set_xlabel('Gaussian standard deviations')
    ax.set_title(f'Z = {Z:.2f}   →   p = {p:.3g}   ({sided}; red area = p)')
    plt.show()

interact(tail_plot, Z=fslider(3, 0, 7, .05, 'Z'), sided=w.ToggleButtons(options=['one-sided', 'two-sided']));

In [ ]:
def from_p(log10_p=-2.0):
    p = 10 ** log10_p
    tail_plot(stats.norm.isf(p))

interact(from_p, log10_p=fslider(-1.3, -9, -0.31, .01, 'log10(p)'));

**The data do not have to be Gaussian.** A counting experiment with a small background $B$ is Poisson.
The $p$-value is computed with the Poisson distribution, $p = P(n \ge n_\text{obs} \mid B)$, and only then converted to $Z$ with the Gaussian convention.

**Try:** with $B = 3$, compare the converted $Z$ with the naive $(n_\text{obs}-B)/\sqrt{B}$. Then set $B = 300$ and $n_\text{obs}$ near $B + 3\sqrt{B} \approx 352$.

In [ ]:
def poisson_pvalue(B=3.0, n_obs=10):
    p = stats.poisson.sf(n_obs - 1, B)                  # P(n >= n_obs | B)
    Z = stats.norm.isf(p) if p > 0 else np.inf
    naive = (n_obs - B) / np.sqrt(B)
    n = np.arange(0, int(max(n_obs, B + 5 * np.sqrt(B))) + 4)
    pmf = stats.poisson.pmf(n, B)
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.bar(n, pmf, width=1, color=np.where(n >= n_obs, DR, GRAY))
    x = np.linspace(0, n[-1], 400); ax.plot(x, stats.norm.pdf(x, B, np.sqrt(B)), '--', color=BB, lw=1.5, label='Gaussian approximation')
    ax.axvline(n_obs - .5, color=DR, lw=1)
    ax.set_xlabel('number of events'); ax.set_yticks([]); ax.legend(fontsize=9)
    ax.set_title(f'Poisson p = {p:.3g}  →  Z = {Z:.2f}      naive (n−B)/√B = {naive:.2f}', fontsize=11)
    plt.show()

interact(poisson_pvalue, B=logslider(3, .1, 1000, 'B (expected background)'), n_obs=islider(10, 0, 1200, 'observed n'));

## Page 8 — Confidence limit

Test the signal-plus-background hypothesis: expected $B+S$ events, standard deviation $\sqrt{B+S}$.
If the observed number of events is $1.64\sigma$ below $B+S$, the probability to observe this few events or fewer is 5%:
this $S$ is excluded at 95% CL. The $S$ for which this happens is the 95% CL upper limit.

**Try:**
- Slide $S$ until the red area is exactly 5%. That $S$ is the upper limit; compare with the printed value.
- Change the CL to 90% or 99%. Does the limit go up or down?
- Make $n_\text{obs}$ much smaller than $B$ (a downward fluctuation). What happens to the limit? (It can even become negative.)
- Use a small $B$ (e.g. 3) and compare the Gaussian and the exact Poisson limit.

In [ ]:
def upper_limits(B, n_obs, CL):
    zc = stats.norm.ppf(CL)
    u = (zc + np.sqrt(zc ** 2 + 4 * n_obs)) / 2            # solve B+S - zc*sqrt(B+S) = n_obs
    lam_up = optimize.brentq(lambda lam: stats.poisson.cdf(n_obs, lam) - (1 - CL), 1e-9, n_obs + 20 + 15 * np.sqrt(n_obs + 1))
    return u ** 2 - B, lam_up - B

def confidence_limit(B=100.0, S=30.0, n_obs=111, CL=0.95):
    mean, sig = B + S, np.sqrt(B + S)
    prob = stats.norm.cdf((n_obs - mean) / sig)
    s_g, s_p = upper_limits(B, n_obs, CL)
    lo = min(B, n_obs, mean) - 4 * sig; hi = max(mean, B + s_g) + 4 * sig
    x = np.linspace(lo, hi, 600); y = np.exp(-(x - mean) ** 2 / (2 * sig ** 2))
    fig, ax = plt.subplots(figsize=(9.5, 3.6))
    ax.fill_between(x, y, color=BB, alpha=.12); ax.plot(x, y, color=BB, lw=2.5, label='$S+B$ hypothesis')
    ax.fill_between(x[x <= n_obs], y[x <= n_obs], color=DR, alpha=.6)
    yu = np.exp(-(x - B - s_g) ** 2 / (2 * (B + s_g))) if B + s_g > 0 else None
    if yu is not None:
        ax.plot(x, yu, ':', color='k', lw=1.3, label=f'hypothesis at the upper limit, S = {s_g:.1f}')
    ax.plot([n_obs, n_obs], [0, 1.1], color=DR, lw=2); ax.text(n_obs, 1.12, 'observed', color=DR, ha='center')
    ax.plot([B], [0], 'o', color=BB, ms=7, clip_on=False); ax.text(B, .05, '$B$', ha='center', color=BB)
    ax.plot([mean, mean], [0, 1], '--', color=BB, lw=1); ax.text(mean, 1.02, '$B+S$', ha='center', color=BB)
    ax.set_ylim(0, 1.3); ax.set_yticks([]); ax.legend(fontsize=8.5, loc='upper right')
    verdict = f'EXCLUDED at {CL:.0%} CL' if prob < 1 - CL else f'not excluded at {CL:.0%} CL'
    ax.set_title(f'red area P(n ≤ n_obs | S+B) = {prob:.3f}  →  S = {S:.0f} {verdict}\n'
                 f'{CL:.0%} CL upper limit on S:  Gaussian {s_g:.1f},  exact Poisson {s_p:.1f}', fontsize=10)
    plt.show()

interact(confidence_limit, B=fslider(100, 1, 400, 1, 'B (expected background)'),
         S=fslider(30, 0, 200, .5, 'S (tested signal)'), n_obs=islider(111, 0, 600, 'observed n'),
         CL=w.Dropdown(options=[0.68, 0.90, 0.95, 0.99], value=0.95, description='CL'));

## Page 9 — Categorization

Inclusive: $Z = \dfrac{s_1+s_2}{\sqrt{b_1+b_2}}$. Categorized: $Z = \sqrt{Z_1^2 + Z_2^2}$, $Z_i = s_i/\sqrt{b_i}$.
As long as $s_1/b_1 \neq s_2/b_2$, the combined significance is **larger**.

Right plot: keep the total signal and both backgrounds fixed, and change how the signal is shared between the two categories.

**Try:**
- Find the split where categorization does not help at all. What is special about it?
- Put almost all signal into a category with little background. How much do you gain?
- Can the categorized $Z$ ever be smaller than the inclusive one?

In [ ]:
def categorization(s1=12.0, b1=18.0, s2=3.0, b2=27.0):
    Z1, Z2 = s1 / np.sqrt(b1), s2 / np.sqrt(b2)
    Zc, Zi = np.hypot(Z1, Z2), (s1 + s2) / np.sqrt(b1 + b2)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.9), gridspec_kw=dict(width_ratios=[1, 1.1]))
    pos, bs, ss = [0, 1.5, 2.3], [b1 + b2, b1, b2], [s1 + s2, s1, s2]
    a1.bar(pos, bs, width=.7, color=GRAY, edgecolor='k'); a1.bar(pos, ss, bottom=bs, width=.7, color=DR, edgecolor='k')
    for p_, b_, s_ in zip(pos, bs, ss):
        a1.text(p_, b_ + s_, f's/b={s_ / b_:.2f}', ha='center', va='bottom', fontsize=9)
    a1.set_xticks(pos); a1.set_xticklabels(['inclusive', 'category 1', 'category 2']); a1.set_ylabel('events')
    a1.set_ylim(0, (b1 + b2 + s1 + s2) * 1.15)
    S = s1 + s2; f = np.linspace(0, 1, 300)
    a2.plot(f, np.hypot(f * S / np.sqrt(b1), (1 - f) * S / np.sqrt(b2)), color=DR, lw=2, label='categorized')
    a2.axhline(Zi, color=BB, lw=2, label='inclusive')
    a2.plot([s1 / S if S > 0 else 0], [Zc], 'o', color=DR, ms=8)
    a2.axvline(b1 / (b1 + b2), ls=':', color='k'); a2.text(b1 / (b1 + b2), Zi * .45, ' $s_1/b_1 = s_2/b_2$', fontsize=9)
    a2.set_xlabel('fraction of signal in category 1,  $s_1/(s_1+s_2)$'); a2.set_ylabel('Z'); a2.legend(fontsize=9)
    a2.set_ylim(0, None)
    fig.suptitle(f'inclusive Z = {Zi:.2f}     categorized Z = √({Z1:.2f}² + {Z2:.2f}²) = {Zc:.2f}')
    plt.tight_layout(); plt.show()

interact(categorization, s1=fslider(12, 0, 60, .5, 's1'), b1=fslider(18, 1, 100, .5, 'b1'),
         s2=fslider(3, 0, 60, .5, 's2'), b2=fslider(27, 1, 100, .5, 'b2'));

## Page 10 — Categories are the input to the measurement

The inclusive $m_{\gamma\gamma}$ spectrum is split into three categories with different signal-to-background ratios.
In the end, these distributions are the input to the measurement, not a single distribution. (Illustrative numbers.)

*Separation* = how well the categorization separates signal from background: 0 means every category has the same $s/b$; 1 is the slide.
The numbers count events within $125 \pm 3.2$ GeV ($\pm2\sigma$ of the peak); $Z$ uses the general formula of page 3, because $s/b$ is not small in category 1.

**Try:**
- Move *separation* from 0 to 1 and watch the combined $Z$ grow while the inclusive $Z$ stays the same.
- Tick *pseudo-data* to see what one experiment would actually record. Is the signal visible in category 3?
- Increase the luminosity. How does $Z$ scale?

In [ ]:
def categories(separation=1.0, lumi=1.0, pseudo_data=False, seed=0):
    s, b = category_templates(separation, lumi)
    win = abs(CENT1 - MH) < 2 * SIGMA_M
    si, bi = s[:, win].sum(1), b[:, win].sum(1)
    Zi = z_asimov(si, bi); Zinc = z_asimov(si.sum(), bi.sum())   # general formula from page 3 (s/b is not small in category 1)
    kc = np.random.default_rng(seed).poisson(s + b)      # one pseudo-experiment, per category
    fig = plt.figure(figsize=(11, 5.2)); gs = GridSpec(3, 2, figure=fig, width_ratios=[1.15, 1], hspace=.45)
    panels = [(fig.add_subplot(gs[:, 0]), s.sum(0), b.sum(0), 'inclusive', None)]
    for c in range(3):
        panels.append((fig.add_subplot(gs[c, 1]), s[c], b[c], f'category {c + 1}', c))
    for ax, ss, bb, name, c in panels:
        ax.fill_between(EDGES1, np.r_[bb, bb[-1]], step='post', color=GRAY)
        ax.fill_between(EDGES1, np.r_[bb, bb[-1]], np.r_[bb + ss, bb[-1] + ss[-1]], step='post', color=DR, alpha=.85)
        if pseudo_data:
            k = kc.sum(0) if c is None else kc[c]
            ax.errorbar(CENT1, k, yerr=np.sqrt(k), fmt='o', ms=2.5, color='k', lw=.7)
        lab = name if c is None else f'{name}:  s={si[c]:.1f}, b={bi[c]:.1f},  Z={Zi[c]:.2f}'
        ax.set_title(lab, fontsize=9.5, loc='left'); ax.set_xlim(LO, HI); ax.set_ylim(0, None)
    panels[0][0].set_xlabel(r'$m_{\gamma\gamma}$ [GeV]'); panels[0][0].set_ylabel('events / GeV')
    panels[-1][0].set_xlabel(r'$m_{\gamma\gamma}$ [GeV]')
    fig.suptitle(f'inclusive Z = {Zinc:.2f}      combined Z = √(ΣZᵢ²) = {np.sqrt((Zi ** 2).sum()):.2f}')
    plt.show()

interact(categories, separation=fslider(1, 0, 1, .05, 'separation'), lumi=logslider(1, .1, 20, 'luminosity ×'),
         pseudo_data=w.Checkbox(False, description='pseudo-data'), seed=islider(0, 0, 100, 'random seed'));

## Page 11 — Likelihood analysis

Poisson probability: $P(k\,|\,\lambda) = \dfrac{\lambda^k e^{-\lambda}}{k!}$, with $k$ the observed number of events (**constant**, fixed by the data) and $\lambda$ the expected number (**free**).
Likelihood: the same expression, as a function of $\lambda$: $\mathcal{L}(\lambda) = P(k\,|\,\lambda)$.

Left: for a fixed $\lambda$, the probability of every possible $k$. Right: for the fixed observed $k$, the likelihood as a function of $\lambda$.
The highlighted bar on the left and the dot on the right are **the same number**.

**Try:**
- Move $\lambda$ and watch the red bar and the dot change together.
- Find the $\lambda$ that makes the observed $k$ most probable.
- Change $k$. Where is the maximum of $\mathcal{L}(\lambda)$ now?

In [ ]:
def two_views(k=30, lam=25.0):
    top = max(60, int(k + 4 * np.sqrt(k + 1)) + 5, int(lam + 4 * np.sqrt(lam)) + 5)
    n = np.arange(0, top)
    pmf = stats.poisson.pmf(n, lam); val = stats.poisson.pmf(k, lam)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
    a1.bar(n, pmf, width=.85, color=np.where(n == k, DR, GRAY))
    a1.set_xlabel('k (number of events)'); a1.set_title(f'P(k | λ = {lam:.1f}) for every k', fontsize=10)
    a1.annotate(f'P({k} | {lam:.1f}) = {val:.4f}', xy=(k, val), xytext=(k, max(pmf.max(), val) * 1.12),
                ha='center', color=DR, fontsize=9, arrowprops=dict(arrowstyle='->', color=DR))
    a1.set_ylim(0, pmf.max() * 1.3)
    lg = np.linspace(.05, top, 600); L = stats.poisson.pmf(k, lg)
    a2.plot(lg, L, color=BB, lw=2.5); a2.plot([lam], [val], 'o', color=DR, ms=8)
    a2.axvline(k, ls='--', color='k', lw=1); a2.text(k, L.max() * 1.05, f' maximum at λ = k = {k}', fontsize=9)
    a2.set_xlabel('λ'); a2.set_ylabel(r'$\mathcal{L}(\lambda) = P(k|\lambda)$')
    a2.set_title(f'likelihood for the observed k = {k}', fontsize=10); a2.set_ylim(0, L.max() * 1.18)
    plt.tight_layout(); plt.show()

interact(two_views, k=islider(30, 0, 100, 'observed k'), lam=fslider(25, .5, 100, .5, 'λ (expected)'));

## Page 12 — Negative log-likelihood

$$-\ln\mathcal{L}(\lambda) = \lambda - k\ln\lambda + \ln k!$$
Maximizing $\mathcal{L}$ ⇔ maximizing $\ln\mathcal{L}$ ⇔ minimizing $-\ln\mathcal{L}$, since $\ln$ is monotonically increasing. The minimum is at $\hat\lambda = k$.

**Try:**
- Move the $\lambda$ marker: the maximum of $\mathcal{L}$ and the minimum of $-\ln\mathcal{L}$ happen at the same $\lambda$.
- Untick *include ln k!*. The curve moves down by a constant; the minimum does not move. (That is why the constant is often dropped.)
- Change $k$ (slide: $k = 30$). Does the curve get wider or narrower relative to $k$?

In [ ]:
def nll_scan(k=30, lam=36.0, include_lnkfact=True):
    lo, hi = max(.3, k - 3 * np.sqrt(k + 1) - 2), k + 3 * np.sqrt(k + 1) + 2
    lg = np.linspace(lo, hi, 500)
    const = gammaln(k + 1) if include_lnkfact else 0.0
    nll = lg - k * np.log(lg) + const
    nl = lam - k * np.log(lam) + const
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
    a1.plot(lg, np.exp(-(lg - k * np.log(lg) + gammaln(k + 1))), color=BB, lw=2.5)
    a1.plot([lam], [stats.poisson.pmf(k, lam)], 'o', color=DR, ms=8)
    a1.axvline(k, ls='--', color='k', lw=1); a1.set_xlabel('λ'); a1.set_ylabel(r'$\mathcal{L}(\lambda)$')
    a1.set_title('likelihood: maximum', fontsize=10); a1.set_xlim(lo, hi)
    a2.plot(lg, nll, color=BB, lw=2.5); a2.plot([lam], [nl], 'o', color=DR, ms=8)
    a2.axvline(k, ls='--', color='k', lw=1)
    a2.set_xlabel('λ'); a2.set_ylabel(r'$-\ln\mathcal{L}(\lambda)$'); a2.set_xlim(lo, hi)
    a2.set_title(f'NLL: minimum at λ̂ = k = {k}' + (f'   (ln {k}! = {gammaln(k + 1):.2f})' if include_lnkfact else '   (ln k! dropped)'), fontsize=10)
    plt.tight_layout(); plt.show()

interact(nll_scan, k=islider(30, 1, 200, 'observed k'), lam=fslider(36, .5, 250, .5, 'λ marker'),
         include_lnkfact=w.Checkbox(True, description='include ln k!'));

## Page 13 — Likelihood analysis: signal strength

$$\lambda = \mu\,s + b, \qquad s = \sigma_\text{SM}\cdot\mathcal{B}\cdot L\cdot\epsilon$$
$s$: expected SM signal; $b$: expected background; $\mu = \sigma/\sigma_\text{SM}$ the signal strength ($\mu = 1$: SM).
$\mu$ is the parameter of interest; its best-fit value minimizes
$$-\ln\mathcal{L}(\mu) = (\mu s + b) - k\ln(\mu s + b) + \ln k!$$

Left: $\lambda(\mu)$ is a straight line with intercept $b$ and slope $s$. Where it crosses the observed $k$ is $\hat\mu = (k-b)/s$. Right: the NLL in $\mu$.

**Try:**
- Double the luminosity $L$. What happens to $s$, to the slope, and to the width of the NLL?
- Make $k < b$ (fewer events than the background alone). What is $\hat\mu$?
- Lower the efficiency $\epsilon$. Why does the NLL get wider?

In [ ]:
def signal_strength(xsec_fb=500.0, BR=0.0025, lumi_ifb=100.0, eff=0.04, b=20.0, k=30):
    s = xsec_fb * BR * lumi_ifb * eff
    mu_min = -b / s
    mu_hat = max((k - b) / s, mu_min)
    half = 4 * np.sqrt(max(k, 1)) / s
    mu = np.linspace(max(mu_min + 1e-6 * (1 + abs(mu_min)), mu_hat - half), mu_hat + half, 500)
    lam = mu * s + b
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
    a1.plot(mu, lam, color=BB, lw=2.5, label=r'$\lambda(\mu) = \mu s + b$')
    a1.axhline(k, color=DR, lw=1.5, label=f'observed k = {k}'); a1.axvline(mu_hat, ls='--', color='k', lw=1)
    a1.axvline(0, color='gray', lw=.6); a1.axvline(1, color='gray', lw=.6, ls=':')
    a1.set_xlabel('μ'); a1.set_ylabel('expected events λ'); a1.legend(fontsize=9)
    a1.set_title(f's = σ·B·L·ε = {xsec_fb:.0f}·{BR:.4f}·{lumi_ifb:.0f}·{eff:.3f} = {s:.2f}', fontsize=10)
    a2.plot(mu, nll_poisson(lam, k), color=BB, lw=2.5)
    a2.axvline(mu_hat, ls='--', color='k', lw=1)
    a2.set_xlabel('μ'); a2.set_ylabel(r'$-\ln\mathcal{L}(\mu)$')
    a2.set_title(f'μ̂ = (k − b)/s = ({k} − {b:.0f})/{s:.2f} = {mu_hat:.2f}', fontsize=10)
    plt.tight_layout(); plt.show()

interact(signal_strength, xsec_fb=fslider(500, 10, 2000, 10, 'σ_SM [fb]'),
         BR=logslider(0.0025, 1e-4, 1, 'branching ratio B'), lumi_ifb=logslider(100, 1, 3000, 'luminosity L [1/fb]'),
         eff=fslider(.04, .005, 1, .005, 'efficiency ε'), b=fslider(20, 0, 200, 1, 'background b'),
         k=islider(30, 0, 300, 'observed k'));

## Page 14 — Negative log-likelihood in $\mu$

The same NLL scan as on page 12, with a different parameterization: $\lambda = \mu s + b$ (slide: $k = 30$, $s = 5$, $b = 20$, so $\mu \in [-1, 5]$ is exactly $\lambda \in [15, 45]$).
The top axis on the right plot shows the $\lambda$ that each $\mu$ corresponds to.

**Try:**
- Change $s$ and $b$: the left plot does not change at all, the right one is stretched and shifted.
- Why does a larger $s$ make the $\mu$ curve narrower?

In [ ]:
def nll_two_params(k=30, s=5.0, b=20.0):
    lo, hi = max(.3, k - 3 * np.sqrt(k + 1) - 2), k + 3 * np.sqrt(k + 1) + 2
    lg = np.linspace(lo, hi, 500); nll = nll_poisson(lg, k)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    a1.plot(lg, nll, color=BB, lw=2.5); a1.axvline(k, ls='--', color='k', lw=1)
    a1.set_xlabel('λ'); a1.set_ylabel(r'$-\ln\mathcal{L}$'); a1.set_title(f'in λ:  λ̂ = {k}', fontsize=10)
    mu = (lg - b) / s
    a2.plot(mu, nll, color=DR, lw=2.5); a2.axvline((k - b) / s, ls='--', color='k', lw=1)
    a2.set_xlabel('μ'); a2.set_title(f'in μ (s = {s:g}, b = {b:g}):  μ̂ = {(k - b) / s:.2f}', fontsize=10, pad=24)
    top = a2.secondary_xaxis('top', functions=(lambda m: m * s + b, lambda l: (l - b) / s)); top.set_xlabel('λ', fontsize=9)
    plt.tight_layout(); plt.show()

interact(nll_two_params, k=islider(30, 1, 200, 'observed k'), s=fslider(5, .5, 50, .5, 'signal s'),
         b=fslider(20, 0, 150, 1, 'background b'));

## Page 15 — Likelihood with multiple categories

Each bin in each category is an observation $k_{i,c}$, with expectation $\lambda_{i,c}(\mu) = \mu\,s_{i,c} + b_{i,c}$. One $\mu$ for all bins and categories (it is physics).
$$-\ln\mathcal{L}(\mu) = \sum_c\sum_i \Big[\lambda_{i,c}(\mu) - k_{i,c}\ln\lambda_{i,c}(\mu) + \ln k_{i,c}!\Big]$$

Pseudo-data are generated in the three categories of page 10 (1 GeV bins) with a true $\mu$. Left: data and the best-fit model per category.
Right: $\Delta\mathrm{NLL}$ (each curve shifted so its minimum is 0) for each category alone and for the sum.

**Try:**
- Which category alone constrains $\mu$ best? Is it the one with the most signal events?
- Untick categories and watch the total curve get wider.
- Click through seeds: the per-category $\hat\mu$ jump around a lot, the combined one less.
- Tick *Asimov* (data = expectation, no fluctuations): every curve has its minimum at the true $\mu$.

In [ ]:
def multi_cat(mu_true=1.0, lumi=1.0, seed=0, asimov=False, cat1=True, cat2=True, cat3=True):
    s, b = category_templates(1.0, lumi)
    rng = np.random.default_rng(seed)
    k = (mu_true * s + b) if asimov else rng.poisson(mu_true * s + b).astype(float)
    use = [c for c, on in enumerate([cat1, cat2, cat3]) if on]
    if not use:
        print('Select at least one category.'); return
    su = s[use]; mu_lo = max(-3.0, -np.min(b[use][su > 0] / su[su > 0]) * 0.98)   # keep lambda > 0 in every bin
    grid = np.linspace(mu_lo, 6, 1201)
    def cat_nll(c):   # unused categories may reach lambda <= 0 on this grid: mark those points as NaN
        lam = grid[:, None] * s[c] + b[c]
        ok = (lam > 0).all(1)
        out = np.full(len(grid), np.nan)
        out[ok] = nll_poisson(lam[ok], k[c]).sum(1)
        return out
    nll_c = np.array([cat_nll(c) for c in range(3)])
    total = nll_c[use].sum(0)
    def summary(curve):
        d = curve - np.nanmin(curve); inside = grid[d <= .5]
        return grid[np.nanargmin(curve)], inside.min(), inside.max(), d
    fig = plt.figure(figsize=(11.5, 5.4)); gs = GridSpec(3, 2, figure=fig, width_ratios=[1, 1.1], hspace=.5)
    mu_hat_tot = summary(total)[0]
    for c in range(3):
        ax = fig.add_subplot(gs[c, 0])
        ax.errorbar(CENT1, k[c], yerr=np.sqrt(k[c]), fmt='o', ms=2.5, color='k', lw=.7, alpha=1 if c in use else .25)
        ax.plot(CENT1, b[c], '--', color=GRAY, lw=1.5); ax.plot(CENT1, mu_hat_tot * s[c] + b[c], color=CATC[c], lw=1.8)
        mh, l1, h1, _ = summary(nll_c[c])
        ax.set_title(f'category {c + 1}' + ('' if c in use else ' (not used)') + f':  μ̂ = {mh:.2f}  [{l1:.2f}, {h1:.2f}]',
                     fontsize=9, loc='left', color=CATC[c])
        ax.set_xlim(LO, HI); ax.set_ylim(0, None)
    ax.set_xlabel(r'$m_{\gamma\gamma}$ [GeV]')
    a2 = fig.add_subplot(gs[:, 1])
    for c in range(3):
        a2.plot(grid, summary(nll_c[c])[3], color=CATC[c], lw=1.8, ls='-' if c in use else ':', label=f'category {c + 1}')
    mh, l1, h1, d = summary(total)
    a2.plot(grid, d, color='k', lw=3, label='sum of used categories')
    a2.axhline(.5, color='gray', ls='--', lw=1); a2.axvline(mu_true, color=DR, lw=1, label=f'true μ = {mu_true:g}')
    a2.set_ylim(0, 4); a2.set_xlim(grid[0], grid[-1]); a2.set_xlabel('μ'); a2.set_ylabel('ΔNLL'); a2.legend(fontsize=8.5)
    a2.set_title(f'combined:  μ̂ = {mh:.2f}  (−{mh - l1:.2f} / +{h1 - mh:.2f})', fontsize=10.5)
    plt.show()

interact(multi_cat, mu_true=fslider(1, 0, 3, .1, 'true μ'), lumi=logslider(1, .2, 20, 'luminosity ×', cont=False),
         seed=islider(0, 0, 200, 'random seed'), asimov=w.Checkbox(False, description='Asimov (no fluctuations)'),
         cat1=w.Checkbox(True, description='use category 1'), cat2=w.Checkbox(True, description='use category 2'),
         cat3=w.Checkbox(True, description='use category 3'));

## Page 16 — Signal strength from the NLL

- Central value $\hat\mu$: the minimum of the NLL.
- Uncertainty ($\pm1\sigma$): where the NLL rises by **0.5** above its minimum. (±2σ: rises by 2 = 2²/2.)

A single counting experiment with $\lambda = \mu s + b$. The defaults ($k = 25$, $s = 12.5$, $b = 10$) give $\hat\mu = 1.2$ with about $\pm0.4$, like the slide.
The dashed curve is the parabola with the same curvature at the minimum.

**Try:**
- Make $k$ small (e.g. 3) with $b$ small. The NLL is no longer a parabola: the uncertainties become asymmetric.
- Multiply $k$, $s$, and $b$ by 4 (four times the data). How does the uncertainty change?

In [ ]:
def counting_interval(k, s, b, nsig=1.0):
    # mu_hat and the interval where Delta NLL <= nsig^2/2, for lambda = mu*s + b >= 0.
    t = nsig ** 2 / 2
    if k == 0:
        lam_lo, lam_hi = 0.0, t                      # Delta NLL = lambda when k = 0
    else:
        d = lambda lam: lam - k - k * np.log(lam / k) - t
        lam_lo = optimize.brentq(d, 1e-12, k) if d(1e-12) > 0 else 0.0
        lam_hi = optimize.brentq(d, k, k + 10 * np.sqrt(k) + 10 * t + 10)
    return (k - b) / s, (lam_lo - b) / s, (lam_hi - b) / s

def mu_from_nll(k=25, s=12.5, b=10.0, show_parabola=True):
    mh, l1, h1 = counting_interval(k, s, b, 1)
    _, l2, h2 = counting_interval(k, s, b, 2)
    mh = max(mh, -b / s)
    mu = np.linspace(max(-b / s + 1e-9, l2 - .6 * (mh - l2) - .05), h2 + .6 * (h2 - mh) + .05, 600)
    lam = mu * s + b; lam_hat = max(k, 1e-12)
    d = lam - k * np.log(lam) - (lam_hat - k * np.log(lam_hat)) if k > 0 else lam
    fig, ax = plt.subplots(figsize=(9, 3.9))
    ax.plot(mu, d, color=BB, lw=2.5, label='ΔNLL')
    if show_parabola and k > 0:
        sig = np.sqrt(k) / s
        ax.plot(mu, (mu - mh) ** 2 / (2 * sig ** 2), '--', color='gray', lw=1.5, label=f'parabola, σ = √k/s = {sig:.3f}')
    for lev, lo_, hi_, lab in [(.5, l1, h1, '±1σ'), (2, l2, h2, '±2σ')]:
        ax.axhline(lev, color='gray', lw=.8, ls=':'); ax.text(mu[-1], lev, f' +{lev:g}', va='center', fontsize=9)
        ax.plot([lo_, lo_], [0, lev], color=DR, lw=1.8); ax.plot([hi_, hi_], [0, lev], color=DR, lw=1.8)
        ax.annotate('', xy=(lo_, lev + .12), xytext=(hi_, lev + .12), arrowprops=dict(arrowstyle='<->', color=DR))
        ax.text((lo_ + hi_) / 2, lev + .2, lab, color=DR, ha='center', fontsize=9)
    ax.set_ylim(0, 3.2); ax.set_xlim(mu[0], mu[-1]); ax.set_xlabel('μ'); ax.set_ylabel('ΔNLL'); ax.legend(fontsize=9, loc='upper center')
    ax.set_title(f'μ̂ = {mh:.2f}  −{mh - l1:.2f} / +{h1 - mh:.2f}  (1σ)       2σ: [{l2:.2f}, {h2:.2f}]')
    plt.show()

interact(mu_from_nll, k=islider(25, 0, 400, 'observed k'), s=fslider(12.5, .5, 100, .5, 'signal s'),
         b=fslider(10, 0, 300, .5, 'background b'), show_parabola=w.Checkbox(True, description='show parabola'));

**Does the ±1σ interval really contain the true $\mu$ 68% of the time?** Repeat the experiment many times with a known true $\mu$,
compute the ΔNLL = 0.5 interval in each, and count how often it contains the truth (the *coverage*).

**Try:** compare the coverage for large counts (defaults) and for very small ones ($s = 2$, $b = 0.5$).

In [ ]:
def coverage(mu_true=1.0, s=12.5, b=10.0, n_toys=2000, seed=0):
    rng = np.random.default_rng(seed)
    ks = rng.poisson(mu_true * s + b, n_toys)
    ints = {kk: counting_interval(kk, s, b, 1) for kk in np.unique(ks)}
    mh = np.array([max(ints[kk][0], -b / s) for kk in ks])
    cov = np.mean([ints[kk][1] <= mu_true <= ints[kk][2] for kk in ks])
    fig, ax = plt.subplots(figsize=(9, 3.4))
    vals, cnt = np.unique(mh, return_counts=True)
    if len(vals) <= 80:
        ax.bar(vals, cnt, width=.8 / s, color=BB, alpha=.6)      # mu_hat = (k - b)/s takes discrete values
    else:
        ax.hist(mh, bins=60, color=BB, alpha=.6)
    ax.axvline(mu_true, color=DR, lw=2, label=f'true μ = {mu_true:g}')
    ax.set_xlabel('μ̂ in each pseudo-experiment'); ax.set_ylabel('pseudo-experiments'); ax.legend(fontsize=9)
    ax.set_title(f'coverage of the ±1σ interval: {cov:.1%}  (target 68.3%)      spread of μ̂: {mh.std():.3f}', fontsize=10.5)
    plt.show()

interact(coverage, mu_true=fslider(1, 0, 3, .1, 'true μ', cont=False), s=fslider(12.5, .5, 100, .5, 'signal s', cont=False),
         b=fslider(10, 0, 300, .5, 'background b', cont=False),
         n_toys=choice([100, 1000, 2000, 10000], 2000, 'pseudo-experiments'), seed=islider(0, 0, 100, 'random seed', cont=False));